# Video Subtitle Processor - Google Colab

Bu notebook, Fransızca videoları indirip Türkçe altyazı ekleyerek işler.

## Özellikler:
- YouTube ve M3U8/MP4 video desteği
- OpenAI Whisper ile Fransızca transkripsiyon
- Türkçe çeviri (Google Translate veya DeepL)
- SRT altyazı dosyası oluşturma
- Altyazıları videoya yazma
- GPU desteği (T4 GPU ile çok daha hızlı)


## 1. Gerekli Paketleri Kur


In [ ]:
# Tüm gerekli paketleri kur
%pip install -q openai-whisper deep-translator torch torchaudio numpy yt-dlp faster-whisper deepl tqdm

# FFmpeg kurulumu (Colab'de genelde hazır ama emin olmak için)
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1

print("✅ Paketler kuruldu!")


## 2. GPU Kontrolü


In [ ]:
import torch

cuda_available = torch.cuda.is_available()
if cuda_available:
    print(f"✅ GPU Aktif: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Bellek: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ GPU bulunamadı, CPU kullanılacak (daha yavaş)")
    print("   Runtime > Change runtime type > Hardware accelerator > GPU seçiniz")


## 3. Video Subtitle Processor Dosyasını Yükle

Aşağıdaki seçeneklerden birini kullanın:

### Seçenek A: Dosyayı Colab'a yükle
1. Sol menüden **Files** sekmesine tıklayın
2. **Upload to session storage** butonuna tıklayın
3. `video_subtitle_processor.py` dosyasını yükleyin

### Seçenek B: GitHub'dan indir (eğer repo'da ise)


In [ ]:
# Eğer GitHub repo'nuz varsa:
# !git clone https://github.com/kullanici/repo.git
# %cd repo

# Veya direkt dosyayı indir:
# !wget https://raw.githubusercontent.com/kullanici/repo/main/video_subtitle_processor.py

# Şimdilik dosyayı manuel yüklediğinizi varsayıyoruz
print("📁 video_subtitle_processor.py dosyasını yüklediğinizden emin olun!")


## 4. Processor'ı İçe Aktar ve Ayarla


In [ ]:
import sys
from pathlib import Path
from google.colab import files
from tqdm import tqdm
import time

# video_subtitle_processor.py dosyasını içe aktar
try:
    from video_subtitle_processor import VideoSubtitleProcessor
    print("✅ VideoSubtitleProcessor yüklendi!")
except ImportError:
    print("❌ video_subtitle_processor.py bulunamadı!")
    print("   Lütfen dosyayı yüklediğinizden emin olun.")
    raise

# GPU varsa faster-whisper kullan, yoksa standart whisper
USE_GPU = torch.cuda.is_available()
WHISPER_MODEL = "medium" if USE_GPU else "base"  # GPU varsa daha büyük model

print(f"\n⚙️ Ayarlar:")
print(f"   GPU: {'Aktif' if USE_GPU else 'Pasif'}")
print(f"   Whisper Model: {WHISPER_MODEL}")
print(f"   Faster Whisper: {USE_GPU}")


## 5. Video URL'ini Gir ve İşle


In [ ]:
# Video URL'ini buraya yapıştır
VIDEO_URL = "https://www.youtube.com/watch?v=CGnqBrqDRfU&t=489s"  # Örnek URL

# DeepL API key (opsiyonel - daha iyi çeviri için)
DEEPL_API_KEY = None  # Varsa buraya yapıştır: "your-api-key-here"

print(f"📹 Video URL: {VIDEO_URL}")
print(f"🌐 Çeviri: {'DeepL' if DEEPL_API_KEY else 'Google Translate'}")


In [ ]:
# İşleme başla
print("🚀 İşlem başlatılıyor...\n")

start_time = time.time()

try:
    # Processor'ı başlat
    processor = VideoSubtitleProcessor(
        output_dir="./output",
        whisper_model=WHISPER_MODEL,
        original_dir="./orijinalini",
        translated_dir="./türkçe_altyazı_eklenmiş_halini",
        burn_subtitles=True,
        use_faster_whisper=USE_GPU,  # GPU varsa faster-whisper kullan
        deepl_api_key=DEEPL_API_KEY,
        use_deepl=bool(DEEPL_API_KEY),
        beam_size=5,
        best_of=5
    )
    
    # Video'yu işle
    with processor:
        output_filename = "translated_video.mp4"
        results = processor.process_video(VIDEO_URL, output_filename)
        
        # Sonuçları göster
        elapsed_time = time.time() - start_time
        print(f"\n✅ İşlem tamamlandı! ({elapsed_time/60:.1f} dakika)")
        print(f"\n📁 Çıktı dosyaları:")
        
        if 'original' in results:
            print(f"   Orijinal video: {results['original']}")
        if 'srt' in results:
            print(f"   SRT altyazı: {results['srt']}")
        if 'final' in results:
            print(f"   ✅ Final video (Türkçe altyazılı): {results['final']}")
            
            # Dosya boyutunu göster
            file_size = Path(results['final']).stat().st_size / (1024 * 1024)
            print(f"   📦 Dosya boyutu: {file_size:.1f} MB")
            
except Exception as e:
    print(f"\n❌ Hata oluştu: {str(e)}")
    import traceback
    traceback.print_exc()


## 6. Dosyaları İndir


In [ ]:
# İşlenmiş videoyu indir
if 'final' in results and Path(results['final']).exists():
    print(f"📥 {results['final']} indiriliyor...")
    files.download(results['final'])
    print("✅ İndirme tamamlandı!")
else:
    print("❌ Final video bulunamadı!")


In [ ]:
# SRT altyazı dosyasını da indir (opsiyonel)
if 'srt' in results and Path(results['srt']).exists():
    print(f"📥 {results['srt']} indiriliyor...")
    files.download(results['srt'])
    print("✅ SRT dosyası indirildi!")
else:
    print("ℹ️ SRT dosyası bulunamadı veya oluşturulmadı.")


In [ ]:
# Drive'ı bağla (ilk kez kullanıyorsanız)
# from google.colab import drive
# drive.mount('/content/drive')

# Dosyaları Drive'a kopyala
# import shutil
# if 'final' in results:
#     drive_path = "/content/drive/MyDrive/subtitle_videos/"
#     Path(drive_path).mkdir(parents=True, exist_ok=True)
#     shutil.copy(results['final'], drive_path)
#     print(f"✅ Video Drive'a kaydedildi: {drive_path}")

print("ℹ️ Drive kaydetme kodu yorum satırında. Kullanmak için yorumları kaldırın.")


## İpuçları

1. **GPU Kullanımı**: Runtime > Change runtime type > Hardware accelerator > GPU seçin
2. **Model Seçimi**: GPU varsa `medium` veya `large`, yoksa `base` veya `small` kullanın
3. **DeepL API**: Daha iyi çeviri için [DeepL API key](https://www.deepl.com/pro-api) alabilirsiniz
4. **Uzun Videolar**: Çok uzun videolar için Colab'ın timeout süresine dikkat edin
5. **Dosya Boyutu**: İşlenmiş videolar büyük olabilir, Drive'a kaydetmeyi düşünün
